In [1]:
import numpy as np
import pandas as pd
from skimage.feature import hog
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, recall_score
import joblib
from collections import Counter

In [2]:
data = pd.read_csv("../archive/ascii_character_classification.csv", header=0).sample(frac=.05)
label_counts = Counter(data.iloc[:, 0])
print(label_counts)

Counter({0: 2498, 29: 293, 6: 272, 55: 272, 93: 267, 21: 265, 91: 263, 90: 262, 31: 262, 67: 257, 9: 256, 86: 255, 34: 255, 49: 254, 26: 254, 3: 253, 63: 252, 94: 252, 79: 249, 83: 248, 1: 248, 77: 248, 17: 248, 43: 247, 54: 245, 37: 245, 58: 245, 66: 245, 19: 244, 52: 244, 82: 244, 73: 243, 12: 243, 84: 243, 32: 243, 92: 243, 11: 242, 20: 242, 71: 241, 61: 241, 70: 241, 2: 240, 15: 240, 30: 239, 69: 238, 75: 237, 24: 237, 76: 236, 39: 236, 5: 235, 85: 235, 44: 234, 4: 234, 65: 233, 28: 232, 81: 232, 16: 232, 23: 231, 62: 231, 48: 230, 14: 230, 95: 229, 10: 229, 68: 229, 46: 229, 87: 229, 18: 228, 57: 228, 50: 227, 78: 227, 64: 227, 72: 227, 22: 227, 53: 226, 8: 226, 13: 226, 74: 225, 45: 224, 89: 223, 40: 222, 38: 221, 33: 220, 80: 220, 35: 220, 27: 219, 88: 218, 59: 217, 25: 217, 47: 217, 51: 216, 7: 215, 56: 214, 60: 210, 42: 206, 36: 204, 41: 202})


In [3]:
X = data.iloc[:, 1:].astype("float64")
y = data.iloc[:, 0].astype("float64")

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [4]:
clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)
joblib.dump(clf, '../artifacts/random_forest_model.pkl')

['../artifacts/random_forest_model.pkl']

In [5]:
y_pred = clf.predict(X_train)
train_accuracy = accuracy_score(y_train, y_pred)
y_pred = clf.predict(X_test)
print("test Shape:", X_test.shape)

test_accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average='weighted')
recall = recall_score(y_test, y_pred, average='weighted')

print(f"Train Accuracy: {train_accuracy*100:.4f}%")
print(f"Test Accuracy: {test_accuracy*100:.4f}%")
print(f"F1 Score: {f1*100:.4f}%")
print(f"Recall: {recall*100:.4f}%")

test Shape: (5000, 100)
Train Accuracy: 97.9350%
Test Accuracy: 91.5400%
F1 Score: 90.6901%
Recall: 91.5400%


In [6]:
def extract_hog_features(images):
    hog_features = []
    for image in images:
        image_reshaped = image.reshape((10, 10))
        features = hog(image_reshaped, pixels_per_cell=(2, 2), cells_per_block=(1, 1), feature_vector=True)
        hog_features.append(features)
    return np.array(hog_features)

X_hog = extract_hog_features(np.array(X))
X_train, X_test, y_train, y_test = train_test_split(X_hog, y, test_size=0.2, random_state=42)

In [9]:
clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)
joblib.dump(clf, '../artifacts/random_forest_model_hog.pkl')

['../artifacts/random_forest_model_hog.pkl']

In [10]:
y_pred = clf.predict(X_train)
train_accuracy = accuracy_score(y_train, y_pred)
y_pred = clf.predict(X_test)
print("test Shape:", X_test.shape)

test_accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average='weighted')
recall = recall_score(y_test, y_pred, average='weighted')

print(f"Train Accuracy: {train_accuracy*100:.4f}%")
print(f"Test Accuracy: {test_accuracy*100:.4f}%")
print(f"F1 Score: {f1*100:.4f}%")
print(f"Recall: {recall*100:.4f}%")

test Shape: (5000, 225)
Train Accuracy: 97.8850%
Test Accuracy: 94.1800%
F1 Score: 93.3218%
Recall: 94.1800%


In [11]:
import cv2

def extract_sift_features(images):
    sift = cv2.SIFT_create()
    sift_features = []
    
    for image in images:
        image_reshaped = image.reshape((10, 10)).astype(np.uint8)
        keypoints, descriptors = sift.detectAndCompute(image_reshaped, None)
        
        # If no keypoints are found, use a zero array of the same length as a typical descriptor
        if descriptors is None:
            descriptors = np.zeros((1, sift.descriptorSize()), dtype=np.float32)
        
        # Flatten descriptors and use them as features
        features = descriptors.flatten()
        sift_features.append(features)
    
    return np.array(sift_features)

# Extract SIFT features
X_sift = extract_sift_features(np.array(X))

In [12]:
X_sift = extract_sift_features(np.array(X))

# Since the number of features might vary, we need to ensure consistent feature vector size
# Here, we'll pad with zeros to the maximum descriptor length found
max_len = max(len(f) for f in X_sift)
X_sift = np.array([np.pad(f, (0, max_len - len(f)), 'constant') for f in X_sift])

# Split the data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X_sift, y, test_size=0.2, random_state=42)

In [13]:
clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)
joblib.dump(clf, '../artifacts/random_forest_model_sift.pkl')

['../artifacts/random_forest_model_sift.pkl']

In [14]:
y_pred = clf.predict(X_train)
train_accuracy = accuracy_score(y_train, y_pred)
y_pred = clf.predict(X_test)
print("test Shape:", X_test.shape)

test_accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average='weighted')
recall = recall_score(y_test, y_pred, average='weighted')

print(f"Train Accuracy: {train_accuracy*100:.4f}%")
print(f"Test Accuracy: {test_accuracy*100:.4f}%")
print(f"F1 Score: {f1*100:.4f}%")
print(f"Recall: {recall*100:.4f}%")

test Shape: (5000, 128)
Train Accuracy: 10.0300%
Test Accuracy: 9.8400%
F1 Score: 1.7630%
Recall: 9.8400%
